<a href="https://colab.research.google.com/github/deepak-kumar-da/smart-comment-classifier/blob/main/notebooks/02_data_cleaning_and_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

import os
import re
import json
import pandas as pd
import numpy as np
from collections import Counter

BASE = "/content/drive/MyDrive/Smart_Comment_Classifier"
RAW = f"{BASE}/datasets/raw"
PROCESSED = f"{BASE}/datasets/processed"
REPORTS = f"{BASE}/datasets/reports"

os.makedirs(PROCESSED,exist_ok=True)
os.makedirs(REPORTS,exist_ok=True)

files = {
    "jigsaw":"jigsaw_toxic_train.csv",
    "uci":"uci_youtube_spam.csv",
    "youtube_45k":"youtube_45k_raw.csv",
    "spamid":"spamid_pair_analyzed.csv"
}

datasets = {}

for name,filename in files.items():
  path = os.path.join(RAW,filename)
  # Use semicolon separator for the spamid dataset
  separator = ';' if name == 'spamid' else ','
  datasets[name] = pd.read_csv(path, sep=separator, on_bad_lines='skip')

Mounted at /content/drive


In [ ]:
def normalize_text(text):
  if pd.isna(text):
    return ""
  text = str(text).lower()
  text = re.sub(r"\s+"," ",text)
  return text.strip()

def has_alphanumeric(text):
  if not isinstance(text,str):
    return False
  return any(char.isalnum() for char in text)

def text_quality_flags(text):
  if pd.isna(text) or not str(text).strip():
    return {
        "missing_text":True,
        "emoji_or_symbol_only":False,
        "very_short":True,
        "no_alphanumeric":True
    }

    text = str(text).lower()
    alnum = sum(char.isalnum() for char in text)

    return {
       "missing_text":False,
       "emoji_or_symbol_only":alnum == 0,
       "very_short":len(text) < 2,
       "no_alphanumeric":alnum == 0
    }

In [ ]:
# Jigsaw Dataset

jigsaw = datasets["jigsaw"].copy()

jigsaw_text_col = "comment_text"

jigsaw_labels = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

jigsaw = jigsaw.dropna(subset=[jigsaw_text_col]).copy()

jigsaw["text"] = jigsaw[jigsaw_text_col].astype(str).str.strip()
jigsaw["text_normalized"] = jigsaw["text"].apply(normalize_text)

jigsaw = jigsaw[jigsaw["text_normalized"] != ""].copy()

for col in jigsaw_labels:
  jigsaw[col] = pd.to_numeric(jigsaw[col],errors="coerce")

print("Jigsaw rows after removing empty text",len(jigsaw))
print("Missing label values")
print(jigsaw[jigsaw_labels].isna().sum())

display(jigsaw[["text"]] + jigsaw[jigsaw_labels].head())

Jigsaw rows after removing empty text 159571
Missing label values
toxic            0
severe_toxic     0
obscene          0
threat           0
insult           0
identity_hate    0
dtype: int64


,identity_hate,insult,obscene,severe_toxic,text,threat,toxic
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
159566,NaN,NaN,NaN,NaN,NaN,NaN,NaN
159567,NaN,NaN,NaN,NaN,NaN,NaN,NaN
159568,NaN,NaN,NaN,NaN,NaN,NaN,NaN
159569,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
jigsaw_conflict = (
    jigsaw.groupby("text_normalized")[jigsaw_labels]
    .nunique()
    .gt(1)
    .any(axis=1)
)

conflicting_texts = set(jigsaw_conflict[jigsaw_conflict].index)

jigsaw["label_conflict"] = (
    jigsaw["text_normalized"].isin(conflicting_texts)
)

print("Conflicting unique comments",len(conflicting_texts))
print("Rows belonging to conflicting comments",jigsaw["label_conflict"].sum())

jigsaw_clean = (
    jigsaw[~jigsaw["label_conflict"]]
    .drop_duplicates(subset=["text_normalized"],keep="first")
    .copy()
)

jigsaw_clean = jigsaw_clean.dropna(subset=jigsaw_labels)

print("jigsaw_usable_rows",len(jigsaw_clean))
print("removed from jigsaw",len(jigsaw) - len(jigsaw_clean))

Conflicting unique comments 41
Rows belonging to conflicting comments 84
jigsaw_usable_rows 159272
removed from jigsaw 299


In [ ]:
# UCI Spam Dataset

uci = datasets["uci"].copy()

uci["text"] = uci["CONTENT"].astype("string").str.strip()
uci["text_normalized"] = uci["text"].apply(normalize_text)

uci = uci.dropna(subset=["text"]).copy()
uci = uci[uci["text_normalized"] != ""].copy()

# The dataset uses 'CLASS' instead of 'label'. Map it here:
uci["label"] = pd.to_numeric(uci["CLASS"], errors="coerce").astype(int)

print("UCI label distribution")
print(uci["label"].value_counts(dropna=False))

uci_conflict = (
    uci.groupby("text_normalized")["label"]
    .nunique()
    .gt(1)
)

uci_conflicting_texts = set(uci_conflict[uci_conflict].index)

uci_clean = (
    uci[~uci["text_normalized"].isin(uci_conflicting_texts)]
    .drop_duplicates(subset=["text_normalized"],keep="first")
    .copy()
)

print("Conflicting comments",len(uci_conflicting_texts))
print("UCI usable rows",len(uci_clean))
datasets["uci_clean"] = uci_clean

UCI label distribution
label
1    1005
0     951
Name: count, dtype: int64
Conflicting comments 0
UCI usable rows 1741


In [ ]:
# Youtube 45k dataset

yt = datasets["youtube_45k"].copy()

yt["text"] = yt["comment_text"].astype("string").str.strip()
yt["text_normalized"] = yt["text"].apply(normalize_text)

yt = yt.dropna(subset=["text"]).copy()
yt = yt[yt["text_normalized"] != ""].copy()

yt_en = yt[
    yt["language"].astype("string").str.lower().eq("en")
].copy()

print("Original Youtube 45k",len(yt))
print("English rows",len(yt_en))
print("Excluded non english rows",len(yt)-len(yt_en))
print("\nSpam Labels")
print(yt_en["label_spam"].value_counts(dropna=False))
print("\nToxicity Labels")
print(yt_en["label_toxicity"].value_counts(dropna=False))

Original Youtube 45k 45000
English rows 15100
Excluded non english rows 29900

Spam Labels
label_spam
not spam    14792
spam          308
Name: count, dtype: int64

Toxicity Labels
label_toxicity
non-toxic    14318
toxic          782
Name: count, dtype: int64


In [ ]:
def prepare_binary_task(df,label_col,task_name):
  temp = df[["text","text_normalized",label_col]].copy()
  temp[label_col] = temp[label_col].astype("string").str.strip()

  temp = temp.dropna(subset=[label_col])
  temp = temp[temp[label_col] != ""]

  conflicts = (
      temp.groupby("text_normalized")[label_col]
      .nunique()
      .gt(1)
  )

  conflicting_texts = set(conflicts[conflicts].index)

  clean = temp[
      ~temp["text_normalized"].isin(conflicting_texts)
  ].drop_duplicates(
      subset = ["text_normalized"],
      keep = "first"
  ).copy()

  clean = clean.rename(columns={label_col:"label"})

  print(f"\n{task_name}")
  print("Conflicting comments",len(conflicting_texts))
  print("usable rows",len(clean))
  print("label distribution")
  print(clean["label"].value_counts())

  return clean

youtube_spam = prepare_binary_task(yt_en,"label_spam","Youtube Spam")
youtube_toxicity = prepare_binary_task(yt_en,"label_toxicity","Youtube Toxicity")


Youtube Spam
Conflicting comments 2
usable rows 14032
label distribution
label
not spam    13787
spam          245
Name: count, dtype: Int64

Youtube Toxicity
Conflicting comments 0
usable rows 14034
label distribution
label
non-toxic    13258
toxic          776
Name: count, dtype: Int64


In [ ]:
# Re-defining the text_quality_flags function with fixed indentation
def text_quality_flags(text):
  if pd.isna(text) or not str(text).strip():
    return {
        "missing_text": True,
        "emoji_or_symbol_only": False,
        "very_short": True,
        "no_alphanumeric": True
    }

  text_str = str(text).lower()
  alnum = sum(char.isalnum() for char in text_str)

  return {
     "missing_text": False,
     "emoji_or_symbol_only": alnum == 0,
     "very_short": len(text_str) < 2,
     "no_alphanumeric": alnum == 0
  }

for name, df in {
    "jigsaw": jigsaw_clean,
    "uci": uci_clean,
    "youtube_spam": youtube_spam,
    "youtube_toxicity": youtube_toxicity
}.items():

    flags = df["text"].apply(text_quality_flags).apply(pd.Series)

    print(f"\n{name.upper()}")
    print(flags.sum())

    df.loc[:, "emoji_or_symbol_only"] = flags["emoji_or_symbol_only"].values
    df.loc[:, "very_short"] = flags["very_short"].values
    df.loc[:, "no_alphanumeric"] = flags["no_alphanumeric"].values

spamid = datasets["spamid"].copy()

spamid["text"] = spamid["comment"].astype("string")
spamid_flags = spamid["text"].apply(text_quality_flags).apply(pd.Series)

print("\nSPAMID")
print(spamid_flags.sum())


JIGSAW
missing_text            0
emoji_or_symbol_only    0
very_short              0
no_alphanumeric         0
dtype: int64

UCI
missing_text            0
emoji_or_symbol_only    6
very_short              0
no_alphanumeric         6
dtype: int64

YOUTUBE_SPAM
missing_text            0
emoji_or_symbol_only    0
very_short              1
no_alphanumeric         0
dtype: int64

YOUTUBE_TOXICITY
missing_text            0
emoji_or_symbol_only    0
very_short              1
no_alphanumeric         0
dtype: int64

SPAMID
missing_text              126
emoji_or_symbol_only    24374
very_short               4984
no_alphanumeric         24500
dtype: int64


In [ ]:
# save all outputs

outputs = {
    "jigsaw_toxicity_clean.csv":jigsaw_clean,
    "uci_spam_clean.csv":uci_clean,
    "youtube45k_spam_english.csv":youtube_spam,
    "youtube45k_toxicity_english.csv":youtube_toxicity,
}

for filename,df in outputs.items():
  path = os.path.join(PROCESSED,filename)
  df.to_csv(path,index=False)
  print("Saved",path,"| Shape",df.shape)

Saved /content/drive/MyDrive/Smart_Comment_Classifier/datasets/processed/jigsaw_toxicity_clean.csv | Shape (159272, 14)
Saved /content/drive/MyDrive/Smart_Comment_Classifier/datasets/processed/uci_spam_clean.csv | Shape (1741, 10)
Saved /content/drive/MyDrive/Smart_Comment_Classifier/datasets/processed/youtube45k_spam_english.csv | Shape (14032, 6)
Saved /content/drive/MyDrive/Smart_Comment_Classifier/datasets/processed/youtube45k_toxicity_english.csv | Shape (14034, 6)


In [ ]:
# generate cleaning report

cleaning_report = {}

for filename,df in outputs.items():
  cleaning_report[filename] = {
      "rows":int(len(df)),
      "columns":int(len(df.columns)),
      "unique_comments":int(df["text_normalized"].nunique()),
      "duplicate_normalized_comments":int(
          df["text_normalized"].duplicated().sum()
      ),
      "emoji_or_symbol_only":int(
          df["emoji_or_symbol_only"].sum()
      ),
      "very_short":int(
          df["very_short"].sum()
      )
  }

report_path = os.path.join(REPORTS,"notebook_02_cleaning_report.json")
with open(report_path,"w") as f:
  json.dump(cleaning_report,f,indent=4)

print(json.dumps(cleaning_report,indent=4))
print("\nReport saved: ",report_path)

{
    "jigsaw_toxicity_clean.csv": {
        "rows": 159272,
        "columns": 14,
        "unique_comments": 159272,
        "duplicate_normalized_comments": 0,
        "emoji_or_symbol_only": 0,
        "very_short": 0
    },
    "uci_spam_clean.csv": {
        "rows": 1741,
        "columns": 10,
        "unique_comments": 1741,
        "duplicate_normalized_comments": 0,
        "emoji_or_symbol_only": 6,
        "very_short": 0
    },
    "youtube45k_spam_english.csv": {
        "rows": 14032,
        "columns": 6,
        "unique_comments": 14032,
        "duplicate_normalized_comments": 0,
        "emoji_or_symbol_only": 0,
        "very_short": 1
    },
    "youtube45k_toxicity_english.csv": {
        "rows": 14034,
        "columns": 6,
        "unique_comments": 14034,
        "duplicate_normalized_comments": 0,
        "emoji_or_symbol_only": 0,
        "very_short": 1
    }
}

Report saved:  /content/drive/MyDrive/Smart_Comment_Classifier/datasets/reports/notebook_02_clean

In [ ]:
jigsaw_final = jigsaw_clean[["text"] + jigsaw_labels].copy()
jigsaw_final["source"] = "jigsaw"
jigsaw_final["language"] = "en"

uci_final = uci_clean[["text"] + ["label"]].copy()
uci_final["source"] = "uci"
uci_final["language"] = "en"

youtube_spam_final = youtube_spam[["text"] + ["label"]].copy()
youtube_spam_final["source"] = "youtube_spam"
youtube_spam_final["language"] = "en"

youtube_toxicity_final = youtube_toxicity[["text"] + ["label"]].copy()
youtube_toxicity_final["source"] = "youtube_toxicity"
youtube_toxicity_final["language"] = "en"

final_datasets = {
    "jigsaw_toxicity_final.csv":jigsaw_final,
    "uci_spam_final.csv":uci_final,
    "youtube_spam_final.csv":youtube_spam_final,
    "youtube_toxicity_final.csv":youtube_toxicity_final
}

for filename,df in final_datasets.items():
  path = os.path.join(PROCESSED,filename)
  df.to_csv(path,index=False)
  print(filename,df.shape)


jigsaw_toxicity_final.csv (159272, 9)
uci_spam_final.csv (1741, 4)
youtube_spam_final.csv (14032, 4)
youtube_toxicity_final.csv (14034, 4)


In [ ]:
spamid_final = datasets["spamid"].copy()

spamid_final = spamid_final[
    ["comment","posting","emoji","spam"]
].copy()

spamid_final = spamid_final.rename(columns={
    "comment":"text",
    "posting":"context",
    "spam":"label"
})

print("before cleaning",spamid_final.shape)
print(spamid_final.isna().sum())

spamid_final = spamid_final.dropna(subset=["text","label"]).copy()

spamid_final["text"] = spamid_final["text"].astype("string").str.strip()
spamid_final = spamid_final[spamid_final["text"] != ""]

spamid_final["label"] = pd.to_numeric(spamid_final["label"],errors="coerce")

spamid_final = spamid_final.dropna(subset=["label"])

spamid_final["source"] = "spamid_pair"
spamid_final["language"] = "id"

print("after cleaning",spamid_final.shape)
print(spamid_final["label"].value_counts())


spamid_final.to_csv(os.path.join(PROCESSED,"spamid_indonesian_separate.csv"),index=False)

before cleaning (72999, 4)
text       125
context    125
emoji      125
label      125
dtype: int64
after cleaning (72873, 6)
label
0.0    54875
1.0    17998
Name: count, dtype: int64


In [ ]:
spamid_final

,text,context,emoji,label,source,language
0,Rambutnya 😍,Photoshoot for US @USER Print magazine and dig...,1.0,0.0,spamid_pair,id
1,"Kereeeeen bangeeeet siiih, aku kirim voice not...",Photoshoot for US @USER Print magazine and dig...,1.0,0.0,spamid_pair,id
2,Couldn't be more proud🔥. LIT!!,Photoshoot for US @USER Print magazine and dig...,1.0,0.0,spamid_pair,id
3,Ayeeeeeeeeeeeeeeeeeee🎉🎉🎉🎉,Photoshoot for US @USER Print magazine and dig...,1.0,0.0,spamid_pair,id
4,🔥😍😍😍😍,Photoshoot for US @USER Print magazine and dig...,1.0,0.0,spamid_pair,id
...,...,...,...,...,...,...
72869,Owj,Live @USER Kamis,0.0,0.0,spamid_pair,id
72870,"Masha Allah..ada aa,indra😍",Live @USER Kamis,1.0,0.0,spamid_pair,id
72871,Alwi lagi dong bintang tamu nya,Live @USER Kamis,0.0,0.0,spamid_pair,id
72872,Sehat slalu 🙏🙏 Ayah,Live @USER Kamis,1.0,0.0,spamid_pair,id
